# LAAPP Multi-Batch Controller Experiment

**Purpose:** Compare adaptive LAAPP with a genuinely fixed-ρ baseline over several batches while keeping the current experiment notebook and CSV results unchanged.

This is an experimental prototype. Results do not establish a formal privacy guarantee or prove novelty by themselves.

Run the cells from top to bottom. Upload `laapp_controller_v2.py` into the Google Drive `LAAPP` folder first. This version updates ρ only for adaptive strategies; the fixed-ρ baseline state remains unchanged. The experiment uses a sample from `person_100000.csv` and does not modify the original dataset.


In [ ]:
from pathlib import Path
import importlib
import re
import time
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/LAAPP')
CSV_PATH = PROJECT_DIR / 'person_100000.csv'
CONTROLLER_PATH = PROJECT_DIR / 'laapp_controller_v2.py'

print('Dataset exists:', CSV_PATH.exists())
print('Controller exists:', CONTROLLER_PATH.exists())
if not CSV_PATH.exists():
    raise FileNotFoundError(f'Dataset not found: {CSV_PATH}')
if not CONTROLLER_PATH.exists():
    raise FileNotFoundError(f'Controller not found: {CONTROLLER_PATH}')


## 1. Load the controller and a manageable sample

Use 300 rows split into six batches of 50. This keeps the first multi-batch run small enough to inspect quickly.


In [ ]:
import sys
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import laapp_controller_v2
importlib.reload(laapp_controller_v2)

from laapp_controller_v2 import (
    ControllerState,
    build_record_text,
    evaluate_spans,
    run_batch,
)

df = pd.read_csv(CSV_PATH, nrows=300)
print('Loaded rows:', len(df))
print('Columns:', list(df.columns))
display(df.head(3))


## 2. Prepare text and character-level reference spans

Each record is converted to text from the known PII columns. Reference spans are generated from the original field values, so the measured scores reflect this structured synthetic dataset and this evaluation procedure.


In [ ]:
PII_COLUMNS = [
    'firstname', 'lastname', 'street', 'streetnumber',
    'address_unit', 'postalcode', 'city', 'phone', 'email'
]

texts = []
gold_spans = []

def find_all_occurrences(text, value):
    value = str(value).strip()
    if not value:
        return []
    spans = []
    start = 0
    while True:
        idx = text.find(value, start)
        if idx < 0:
            break
        spans.append((idx, idx + len(value)))
        start = idx + max(1, len(value))
    return spans

for _, row in df.iterrows():
    text = build_record_text(row)
    row_spans = []
    for col in PII_COLUMNS:
        if col not in df.columns or pd.isna(row[col]):
            continue
        value = str(row[col]).strip()
        if value:
            row_spans.extend(find_all_occurrences(text, value))
    texts.append(text)
    gold_spans.append(sorted(set(row_spans)))

print('Prepared records:', len(texts))
print('Example text:', texts[0][:250] if texts else 'No records')


## 3. Initialize Presidio

The analyzer is used only for records selected for deep detection. If the package is not installed, this cell installs the analyzer package.


In [ ]:
try:
    from presidio_analyzer import AnalyzerEngine
except ImportError:
    %pip -q install presidio-analyzer
    from presidio_analyzer import AnalyzerEngine

analyzer = AnalyzerEngine()
print('Presidio analyzer ready')


## 4. Run six batches through two comparable strategies

- **Static-rho-fail-closed:** fixed ρ of 0.30 in every batch, with heuristic fallback masking.
- **LAAPP:** adaptive ρ state carried from one batch to the next, with heuristic fallback masking.
- **LAAPP-no-fail-closed:** same adaptive controller, but no fallback; included to show the fallback's effect.

The adaptive controller updates its ρ using observed batch latency. Because Presidio latency varies by runtime and hardware, run this more than once before drawing conclusions.


In [ ]:
BATCH_SIZE = 50
LATENCY_LIMIT_SECONDS = 0.55

strategy_specs = [
    ('Static-rho-fail-closed', ControllerState(
        rho=0.30, latency_limit_s=LATENCY_LIMIT_SECONDS
    )),
    ('LAAPP', ControllerState(
        rho=0.30, latency_limit_s=LATENCY_LIMIT_SECONDS
    )),
    ('LAAPP-no-fail-closed', ControllerState(
        rho=0.30, latency_limit_s=LATENCY_LIMIT_SECONDS
    )),
]

batch_summaries = []
batch_metrics = []

for strategy, state in strategy_specs:
    print(f'Running strategy: {strategy}')
    for batch_idx, start in enumerate(range(0, len(texts), BATCH_SIZE), start=1):
        end = min(start + BATCH_SIZE, len(texts))
        batch_texts = texts[start:end]
        batch_gold = gold_spans[start:end]

        records, summary = run_batch(
            texts=batch_texts,
            analyzer=analyzer,
            state=state,
            strategy=strategy,
        )
        metrics = evaluate_spans(records, batch_gold)

        summary['batch'] = batch_idx
        summary['row_start'] = start
        summary['row_end_exclusive'] = end
        batch_summaries.append(summary)

        metrics['strategy'] = strategy
        metrics['batch'] = batch_idx
        metrics['records'] = len(batch_texts)
        batch_metrics.append(metrics)

summary_df = pd.DataFrame(batch_summaries)
metrics_df = pd.DataFrame(batch_metrics)

print('Batch runs completed:', len(summary_df))
display(summary_df[[
    'strategy', 'batch', 'records', 'batch_latency_s',
    'avg_latency_ms', 'throughput_records_s',
    'escalated_records', 'escalation_rate',
    'fail_closed_records', 'rho_after_update'
]])


## 5. Compare aggregate results

These aggregate rows summarize the batches. Compare recall/leakage and latency together; do not treat any single metric as proof of privacy safety.


In [ ]:
aggregate_rows = []
for strategy in summary_df['strategy'].unique():
    s = summary_df[summary_df['strategy'] == strategy]
    m = metrics_df[metrics_df['strategy'] == strategy]
    aggregate_rows.append({
        'strategy': strategy,
        'batches': len(s),
        'records': int(s['records'].sum()),
        'total_batch_latency_s': round(s['batch_latency_s'].sum(), 4),
        'mean_batch_latency_s': round(s['batch_latency_s'].mean(), 4),
        'mean_escalation_rate': round(s['escalation_rate'].mean(), 4),
        'total_escalated_records': int(s['escalated_records'].sum()),
        'total_fail_closed_records': int(s['fail_closed_records'].sum()),
        'mean_character_recall': round(m['character_recall'].mean(), 4),
        'mean_character_leakage': round(m['character_leakage'].mean(), 4),
        'mean_character_precision': round(m['character_precision'].mean(), 4),
        'mean_overmask_fraction': round(m['overmask_fraction_of_text'].mean(), 4),
        'rho_last_batch': float(s['rho_after_update'].iloc[-1]),
    })

aggregate_df = pd.DataFrame(aggregate_rows)
display(aggregate_df)


## 6. Visualize per-batch behavior

The charts show measured behavior for this run. Timing can fluctuate with Colab load and model warm-up.


In [ ]:
plt.figure(figsize=(10, 5))
for strategy in summary_df['strategy'].unique():
    part = summary_df[summary_df['strategy'] == strategy]
    plt.plot(part['batch'], part['batch_latency_s'], marker='o', label=strategy)
plt.axhline(LATENCY_LIMIT_SECONDS, linestyle='--', label='Latency limit')
plt.xlabel('Batch')
plt.ylabel('Batch latency (seconds)')
plt.title('LAAPP multi-batch latency')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
for strategy in summary_df['strategy'].unique():
    part = summary_df[summary_df['strategy'] == strategy]
    plt.plot(part['batch'], part['escalation_rate'], marker='o', label=strategy)
plt.xlabel('Batch')
plt.ylabel('Escalation rate')
plt.title('Escalation rate across batches')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
for strategy in metrics_df['strategy'].unique():
    part = metrics_df[metrics_df['strategy'] == strategy]
    plt.plot(part['batch'], part['character_recall'], marker='o', label=strategy)
plt.xlabel('Batch')
plt.ylabel('Character recall')
plt.title('Character recall across batches')
plt.legend()
plt.tight_layout()
plt.show()


## 7. Save new results separately

These filenames are new and will not overwrite the earlier single-batch experiment outputs.


In [ ]:
summary_path = PROJECT_DIR / 'laapp_multibatch_summary.csv'
metrics_path = PROJECT_DIR / 'laapp_multibatch_metrics.csv'
aggregate_path = PROJECT_DIR / 'laapp_multibatch_aggregate.csv'

summary_df.to_csv(summary_path, index=False)
metrics_df.to_csv(metrics_path, index=False)
aggregate_df.to_csv(aggregate_path, index=False)

print('Saved:', summary_path)
print('Saved:', metrics_path)
print('Saved:', aggregate_path)


## 8. Interpretation checklist

Before claiming the adaptive controller improves the baseline, check:

1. Does LAAPP change ρ across batches, rather than staying at 0.30?
2. Does it meet the latency limit across batches?
3. Does recall remain acceptable compared with fixed ρ?
4. Is any latency or recall advantage repeatable across multiple runs?
5. Does the result hold on a separate free-text benchmark, not only structured synthetic records?

**Limitations:** This notebook tests a prototype over batches of structured synthetic records converted into text. The fallback masking is heuristic, the risk score is not calibrated, and the measured character-level scores depend on the reference-span construction. This is not a formal privacy guarantee or a novelty proof.
